# RoadSafe Delhi — Linear Regression Model Development (Baseline)
## Assigned Member: Anushree
### Task: Continuous Road Crash Density Estimation & OLS Baseline

---

## 1. Project Introduction & Assigned Member
- **Project:** RoadSafe Delhi — Explainable Road Risk Mapping and Safety Intervention Prioritization Using Machine Learning
- **Assigned Member:** Anushree
- **Model Assignment:** Linear Regression (`sklearn.linear_model.LinearRegression` / `Ridge`)
- **Role:** Continuous baseline regression modeling, feature engineering support, and existing-work domain research (5 papers).


## 2. Problem Statement & Model Objective
The objective is to establish an interpretable Ordinary Least Squares (OLS) / Ridge baseline to predict continuous road risk density (`crash_density_per_km`).
- **Strict Requirement:** Linear Regression must predict a defensible **continuous** target rather than being forced into binary classification.
- **Target Formulation:**
  $$\text{crash\_density\_per\_km} = \frac{\text{crash\_count}}{\text{length\_km}}$$


## 3. Dataset Loading from Common Approved Dataset
Loading the shared spatial partitions from `data_final/`.


In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../../..")) if os.path.basename(os.getcwd()) == "notebooks" else os.path.abspath(".")
DATA_FINAL_DIR = os.path.join(PROJECT_ROOT, "data_final")
ARTIFACTS_DIR = os.path.join(PROJECT_ROOT, "models", "linear_regression", "artifacts")
OUTPUTS_DIR = os.path.join(PROJECT_ROOT, "models", "linear_regression", "outputs")
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

df_train = pd.read_parquet(os.path.join(DATA_FINAL_DIR, "train.parquet"))
df_val = pd.read_parquet(os.path.join(DATA_FINAL_DIR, "validation.parquet"))
df_test = pd.read_parquet(os.path.join(DATA_FINAL_DIR, "test.parquet"))

print(f"Loaded Train: {len(df_train):,}, Val: {len(df_val):,}, Test: {len(df_test):,}")


## 4. Target Variable Selection (Continuous Risk Rate)
Selected Continuous Target: `crash_density_per_km`.


In [ ]:
target_col = "crash_density_per_km"
print(f"Target: {target_col}")
print(f"Train Mean Crash Density: {df_train[target_col].mean():.4f} crashes/km")
print(f"Val Mean Crash Density:   {df_val[target_col].mean():.4f} crashes/km")
print(f"Test Mean Crash Density:  {df_test[target_col].mean():.4f} crashes/km")


## 5. Approved Feature Selection
Loading `feature_contract.json` and asserting that zero outcome-derived variables enter features.


In [ ]:
with open(os.path.join(DATA_FINAL_DIR, "feature_contract.json"), "r") as f:
    contract = json.load(f)

num_features = contract["approved_numerical_predictors"]
cat_features = contract["approved_categorical_predictors"]
approved_predictors = num_features + cat_features

forbidden = set(contract["quarantined_leakage_columns"]).union(set(contract["target_definitions"].keys()))
assert len(set(approved_predictors).intersection(forbidden)) == 0, "FATAL: Leakage detected!"
print("Approved Predictors:", approved_predictors)


## 6. Data Preprocessing (Standard Scaling & One-Hot Encoding)
Fitting transformations strictly on the training partition.


In [ ]:
scaler = StandardScaler()
scaler.fit(df_train[num_features])

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder.fit(df_train[cat_features])

X_train = np.hstack([scaler.transform(df_train[num_features]), encoder.transform(df_train[cat_features])])
X_val = np.hstack([scaler.transform(df_val[num_features]), encoder.transform(df_val[cat_features])])
X_test = np.hstack([scaler.transform(df_test[num_features]), encoder.transform(df_test[cat_features])])

y_train = df_train[target_col].values
y_val = df_val[target_col].values
y_test = df_test[target_col].values

feature_names = num_features + encoder.get_feature_names_out(cat_features).tolist()
print(f"Prepared {X_train.shape[1]} features for Linear Regression.")


## 7. Train-Validation-Test Data Verification (Common Spatial Split)
Verifying disjoint spatial block boundaries.


In [ ]:
assert set(df_train['spatial_block']).isdisjoint(set(df_val['spatial_block']))
assert set(df_train['spatial_block']).isdisjoint(set(df_test['spatial_block']))
print("Spatial Independence Confirmed.")


## 8. Model Architecture & Configuration
Configuring regularized Ridge Linear Regression to prevent numerical collinearity instability.


In [ ]:
lr_model = Ridge(alpha=10.0, random_state=42)
print("Configured Linear Model:", lr_model)


## 9. Model Training
Fitting linear model on `X_train`, `y_train`.


In [ ]:
lr_model.fit(X_train, y_train)
print("Linear Regression Model Fitted.")


## 10. Validation Tuning
Assessing alpha regularization parameter on validation set.


In [ ]:
y_val_pred = lr_model.predict(X_val)
val_mse = mean_squared_error(y_val, y_val_pred)
val_mae = mean_absolute_error(y_val, y_val_pred)
val_r2 = r2_score(y_val, y_val_pred)

print(f"Validation MAE:  {val_mae:.4f}")
print(f"Validation RMSE: {np.sqrt(val_mse):.4f}")
print(f"Validation R²:   {val_r2:.4f}")


## 11. Final Testing on the Untouched Test Dataset
Evaluating OLS / Ridge baseline on the untouched test partition.


In [ ]:
y_test_pred = lr_model.predict(X_test)
test_mse = mean_squared_error(y_test, y_test_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

print(f"Test MAE:  {test_mae:.4f} crashes/km")
print(f"Test RMSE: {np.sqrt(test_mse):.4f} crashes/km")
print(f"Test R²:   {test_r2:.4f}")


## 12. Evaluation Metrics & Visualizations
Plotting Regression Coefficients and Residual Distribution.


In [ ]:
coefs = pd.Series(lr_model.coef_, index=feature_names).sort_values()
plt.figure(figsize=(10, 6))
coefs.plot(kind="barh", color="#2980b9")
plt.title("Ridge Linear Regression Coefficients (Crash Density / km)")
plt.xlabel("Coefficient Weight")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, "coefficients.png"), dpi=200)
plt.show()

# Residuals Plot
residuals = y_test - y_test_pred
plt.figure(figsize=(8, 5))
sns.histplot(residuals.clip(-5, 5), bins=40, kde=True, color="#8e44ad")
plt.title("Residual Error Distribution (Clipped at ±5 crashes/km)")
plt.xlabel("True - Predicted")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, "residuals_distribution.png"), dpi=200)
plt.show()


## 13. Results Interpretation
Analysis of coefficient magnitudes (e.g. carriage lanes, builtup density, distance to center) in predicting continuous crash rate per kilometer.


In [ ]:
print("Top Positive Risk Multipliers:\n", coefs.tail(4))
print("\nTop Negative Mitigating Factors:\n", coefs.head(4))


## 14. Model and Artifact Saving
Saving model, scaler, encoder, and metrics.


In [ ]:
import joblib

joblib.dump(lr_model, os.path.join(ARTIFACTS_DIR, "linear_regression_model.joblib"))
joblib.dump(scaler, os.path.join(ARTIFACTS_DIR, "scaler.joblib"))
joblib.dump(encoder, os.path.join(ARTIFACTS_DIR, "encoder.joblib"))

metrics = {"test_mae": float(test_mae), "test_rmse": float(np.sqrt(test_mse)), "test_r2": float(test_r2)}
with open(os.path.join(OUTPUTS_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Saved Linear Regression artifacts to {ARTIFACTS_DIR}")


## 15. Final Summary & Observations
Summary of linear baseline performance and comparison with deep learning / non-linear methods.
